# XGBoost for Regression

XGBoost for Regression follows the same **stage-wise additive boosting** strategy as Gradient Boosting, but introduces a completely different way of building decision trees. Instead of using traditional impurity measures, XGBoost evaluates splits using **Similarity Scores** and **Gain**, while also incorporating **regularization** to reduce overfitting.

---

# 1. XGBoost vs. Gradient Boosting

The overall boosting workflow remains identical to Gradient Boosting.

1. Initialize a base prediction.
2. Compute residuals.
3. Train a decision tree on the residuals.
4. Update predictions using the learning rate.
5. Repeat until the desired number of trees has been built.

The key difference lies in **tree construction**.

| Gradient Boosting | XGBoost |
|-------------------|----------|
| Standard Decision Trees | Optimized Decision Trees |
| Uses MSE for splitting | Uses Similarity Score & Gain |
| Little regularization | Built-in regularization |
| Slower implementation | Highly optimized implementation |

---

# 2. Stage 1 — Initial Base Prediction

For regression, the first model predicts the **mean of the target values**.

$$
F_0(x)=\bar{y}
$$

### Example

Suppose the average package of all students is

$$
F_0(x)=7.3
$$

Every training sample initially receives this prediction.

---

# 3. Compute Initial Residuals

Residuals measure the prediction error.

$$
\boxed{
R_1=y-F_0(x)
}
$$

### Example

| Actual Package | Prediction | Residual |
|---------------:|-----------:|----------:|
| 4.5 | 7.3 | -2.8 |
| 11.0 | 7.3 | 3.7 |
| 6.0 | 7.3 | -1.3 |
| 8.0 | 7.3 | 0.7 |

These residuals become the target values for the first regression tree.

---

# 4. Tree Construction in XGBoost

Unlike Gradient Boosting, XGBoost does **not** split nodes using MSE.

Instead, every node is evaluated using the **Similarity Score (SS)**.

## Similarity Score

$$
\boxed{
\text{Similarity Score}
=
\frac{\left(\sum \text{Residuals}\right)^2}
{n+\lambda}
}
$$

where

- $n$ = Number of samples in the node
- $\lambda$ = L2 Regularization parameter

---

## Example

Residuals

$$
-2.8,\;3.7,\;-1.3,\;0.7
$$

Their sum is

$$
0.3
$$

Therefore,

$$
\text{Root SS}
=
\frac{(0.3)^2}{4+0}
=
0.0225
$$

---

# 5. Finding the Best Split

Each feature is sorted.

Possible split thresholds are generated by taking the midpoint between adjacent feature values.

For CGPA

$$
5.85,\;
7.10,\;
8.25
$$

For every split, XGBoost computes the **Gain**.

---

## Gain Formula

$$
\boxed{
\text{Gain}
=
SS_{\text{Left}}
+
SS_{\text{Right}}
-
SS_{\text{Parent}}
}
$$

The split with the **highest Gain** is selected.

---

### Example

| Split | Gain |
|-------|------:|
| CGPA < 5.85 | 0.52 |
| CGPA < 7.10 | 5.06 |
| CGPA < 8.25 | 17.52 |

Since

$$
17.52
>
5.06
>
0.52
$$

the split

$$
\boxed{
CGPA<8.25
}
$$

is selected.

This process continues recursively until stopping conditions such as **max depth** are reached.

---

# 6. Computing Leaf Output Values

After the tree structure is complete, every terminal leaf receives an output value.

Unlike traditional regression trees, XGBoost computes

$$
\boxed{
\text{Leaf Output}
=
\frac{\sum \text{Residuals}}
{n+\lambda}
}
$$

where

- $\sum \text{Residuals}$ = Sum of residuals inside the leaf
- $n$ = Number of samples inside the leaf
- $\lambda$ = L2 Regularization

When

$$
\lambda=0
$$

this becomes

$$
\text{Leaf Output}
=
\text{Mean Residual}
$$

---

# 7. Update the Ensemble

The newly built tree is scaled using the learning rate.

$$
\boxed{
F_1(x)
=
F_0(x)
+
\eta\,
\text{Tree}_1(x)
}
$$

where

- $\eta$ = Learning Rate
- Typical default

$$
\eta=0.3
$$

A smaller learning rate means slower but more stable learning.

---

# 8. Compute New Residuals

After updating predictions,

$$
\boxed{
R_2
=
y-F_1(x)
}
$$

The new residuals are smaller than the previous ones because the model has improved.

These updated residuals become the training targets for the next tree.

---

# 9. Repeat the Process

For every boosting iteration,

1. Compute residuals.
2. Build a new tree.
3. Compute Similarity Scores.
4. Compute Gain.
5. Select the best splits.
6. Compute leaf outputs.
7. Update predictions.

The ensemble after $M$ trees becomes

$$
\boxed{
F_M(x)
=
F_0(x)
+
\eta
\sum_{m=1}^{M}
\text{Tree}_m(x)
}
$$

As more trees are added, the residuals approach zero.

---

# 10. Tree Construction Algorithms

XGBoost provides multiple methods for finding split points.

## Exact Greedy Algorithm

- Evaluates every possible split.
- Produces the optimal split.
- Computationally expensive.
- Best suited for smaller datasets.

---

## Approximate Algorithm

- Uses quantile sketches.
- Divides continuous values into bins.
- Evaluates only representative split points.
- Much faster for large datasets.

---

# 11. Summary Table

| Concept | Description |
|---------|-------------|
| Initial Prediction | Mean of target values |
| Residual | $R=y-\hat{y}$ |
| Weak Learner | Regression Tree |
| Similarity Score | $\frac{(\sum Residuals)^2}{n+\lambda}$ |
| Gain | $SS_{Left}+SS_{Right}-SS_{Parent}$ |
| Best Split | Split with highest Gain |
| Leaf Output | $\frac{\sum Residuals}{n+\lambda}$ |
| Learning Rate | $\eta$ |
| Ensemble Update | $F_m(x)=F_{m-1}(x)+\eta\cdot Tree_m(x)$ |
| Exact Greedy | Evaluates every split |
| Approximate Algorithm | Uses quantile-based candidate splits |

---

# Key Takeaways

- XGBoost follows the same boosting pipeline as Gradient Boosting.
- The major innovation lies in **Similarity Scores** and **Gain** for tree construction.
- Regularization ($\lambda$) is built directly into the tree-building process.
- Every tree predicts residuals from the previous ensemble.
- Leaf values are computed using a regularized formula rather than simple averages.
- Predictions are updated using the learning rate.
- XGBoost supports both **Exact Greedy** and **Approximate** split-finding algorithms for efficient training on datasets of different sizes.